In [1]:
import os
import sys
import shutil
import time
import pandas as pd
from datetime import datetime, timedelta

# Menyelaraskan Python worker dengan kernel aktif
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, window, sum as spark_sum, count

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas10_KopiNusantara_Streaming") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print(f"SparkSession aktif. Versi Spark: {spark.version}")

26/11/11 20:06:01 WARN Utils: Your hostname, revan18-1701 resolves to a loopback address: 127.0.1.1; using 192.168.1.199 instead (on interface wlp2s0)
26/11/11 20:06:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/11 20:06:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession aktif. Versi Spark: 3.5.9


In [2]:
folder_stream = "tugas10_stream"
shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# Definisi skema eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membaca stream data CSV
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Streaming source berhasil disiapkan.")
print("df_stream.isStreaming:", df_stream.isStreaming)
df_stream.printSchema()

Streaming source berhasil disiapkan.
df_stream.isStreaming: True
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



In [3]:
# Menambahkan kolom total_harga
df_stream_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi total pendapatan dan jumlah pesanan per cabang
agregasi_cabang = df_stream_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Memulai streaming query
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query 'agregasi_cabang' berhasil dijalankan.")
print("Status aktif:", query_cabang.isActive)

Streaming query 'agregasi_cabang' berhasil dijalankan.
Status aktif: True


In [4]:
# Membaca stream kedua dari folder yang sama
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

df_stream_waktu_total = df_stream_waktu.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi window 2 menit berdasarkan kolom waktu_pesan
agregasi_waktu = df_stream_waktu_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Memulai streaming query
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query 'agregasi_waktu' berhasil dijalankan.")
print("Status aktif:", query_waktu.isActive)

Streaming query 'agregasi_waktu' berhasil dijalankan.
Status aktif: True


In [5]:
# Menentukan waktu referensi untuk pengujian window waktu
waktu_dasar = datetime.now().replace(microsecond=0)

# Batch 1: 4 pesanan di Magelang, Yogyakarta, dan Semarang
batch1 = pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003", "P004"],
    "nama_cabang": [
        "Kopi Nusantara Magelang",
        "Kopi Nusantara Yogyakarta",
        "Kopi Nusantara Semarang",
        "Kopi Nusantara Magelang"
    ],
    "nama_menu": ["Espresso", "Latte", "Cappuccino", "Kopi Tubruk"],
    "qty": [2, 1, 2, 3],
    "harga_satuan": [15000, 26000, 25000, 12000],
    "waktu_pesan": [
        (waktu_dasar - timedelta(minutes=4)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar - timedelta(minutes=4)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar - timedelta(minutes=2)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar - timedelta(minutes=2)).strftime("%Y-%m-%d %H:%M:%S")
    ]
})

batch1.to_csv(f"{folder_stream}/batch_1.csv", index=False)
print("Batch 1 terkirim. Menunggu Spark memproses...")
time.sleep(7)

print("=== Hasil Agregasi Cabang (Setelah Batch 1) ===")
spark.sql("""
    SELECT nama_cabang, total_pendapatan, jumlah_pesanan 
    FROM agregasi_cabang 
    ORDER BY total_pendapatan DESC
""").show(truncate=False)

Batch 1 terkirim. Menunggu Spark memproses...


=== Hasil Agregasi Cabang (Setelah Batch 1) ===


+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Magelang  |66000           |2             |
|Kopi Nusantara Semarang  |50000           |1             |
|Kopi Nusantara Yogyakarta|26000           |1             |
+-------------------------+----------------+--------------+



In [6]:
# Batch 2: 4 pesanan baru yang menambah cabang Solo dan menambah transaksi cabang lain
batch2 = pd.DataFrame({
    "pesanan_id": ["P005", "P006", "P007", "P008"],
    "nama_cabang": [
        "Kopi Nusantara Magelang",
        "Kopi Nusantara Solo",
        "Kopi Nusantara Yogyakarta",
        "Kopi Nusantara Semarang"
    ],
    "nama_menu": ["Americano", "Mocha", "Roti Bakar Cokelat", "Ayam Geprek"],
    "qty": [2, 1, 2, 1],
    "harga_satuan": [18000, 28000, 18000, 25000],
    "waktu_pesan": [
        (waktu_dasar - timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar - timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        waktu_dasar.strftime("%Y-%m-%d %H:%M:%S"),
        waktu_dasar.strftime("%Y-%m-%d %H:%M:%S")
    ]
})

batch2.to_csv(f"{folder_stream}/batch_2.csv", index=False)
print("Batch 2 terkirim. Menunggu Spark memproses...")
time.sleep(7)

print("=== Hasil Agregasi Cabang (Setelah Batch 2 — Akumulasi Terlihat) ===")
spark.sql("""
    SELECT nama_cabang, total_pendapatan, jumlah_pesanan 
    FROM agregasi_cabang 
    ORDER BY total_pendapatan DESC
""").show(truncate=False)

Batch 2 terkirim. Menunggu Spark memproses...


=== Hasil Agregasi Cabang (Setelah Batch 2 — Akumulasi Terlihat) ===
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Magelang  |102000          |3             |
|Kopi Nusantara Semarang  |75000           |2             |
|Kopi Nusantara Yogyakarta|62000           |2             |
|Kopi Nusantara Solo      |28000           |1             |
+-------------------------+----------------+--------------+



In [7]:
# Batch 3: 4 pesanan baru termasuk cabang Purworejo
batch3 = pd.DataFrame({
    "pesanan_id": ["P009", "P010", "P011", "P012"],
    "nama_cabang": [
        "Kopi Nusantara Purworejo",
        "Kopi Nusantara Solo",
        "Kopi Nusantara Magelang",
        "Kopi Nusantara Yogyakarta"
    ],
    "nama_menu": ["Pisang Goreng Keju", "Espresso", "Latte", "Mie Tek-Tek"],
    "qty": [2, 3, 1, 2],
    "harga_satuan": [16000, 15000, 26000, 20000],
    "waktu_pesan": [
        (waktu_dasar + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar + timedelta(minutes=1)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar + timedelta(minutes=3)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar + timedelta(minutes=3)).strftime("%Y-%m-%d %H:%M:%S")
    ]
})

batch3.to_csv(f"{folder_stream}/batch_3.csv", index=False)
print("Batch 3 terkirim. Menunggu Spark memproses...")
time.sleep(7)

print("=== Hasil Agregasi Cabang (Setelah Batch 3 — Akumulasi Akhir) ===")
spark.sql("""
    SELECT nama_cabang, total_pendapatan, jumlah_pesanan 
    FROM agregasi_cabang 
    ORDER BY total_pendapatan DESC
""").show(truncate=False)

Batch 3 terkirim. Menunggu Spark memproses...


=== Hasil Agregasi Cabang (Setelah Batch 3 — Akumulasi Akhir) ===
+-------------------------+----------------+--------------+
|nama_cabang              |total_pendapatan|jumlah_pesanan|
+-------------------------+----------------+--------------+
|Kopi Nusantara Magelang  |128000          |4             |
|Kopi Nusantara Yogyakarta|102000          |3             |
|Kopi Nusantara Semarang  |75000           |2             |
|Kopi Nusantara Solo      |73000           |2             |
|Kopi Nusantara Purworejo |32000           |1             |
+-------------------------+----------------+--------------+



In [8]:
print("=== Hasil Agregasi Berbasis Window Waktu 2 Menit ===")
spark.sql("""
    SELECT 
        window.start AS waktu_mulai, 
        window.end AS waktu_selesai, 
        total_pendapatan, 
        jumlah_pesanan 
    FROM agregasi_waktu 
    ORDER BY window.start ASC
""").show(truncate=False)

=== Hasil Agregasi Berbasis Window Waktu 2 Menit ===


+-------------------+-------------------+----------------+--------------+
|waktu_mulai        |waktu_selesai      |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+----------------+--------------+
|2026-11-11 20:02:00|2026-11-11 20:04:00|56000           |2             |
|2026-11-11 20:04:00|2026-11-11 20:06:00|150000          |4             |
|2026-11-11 20:06:00|2026-11-11 20:08:00|138000          |4             |
|2026-11-11 20:08:00|2026-11-11 20:10:00|66000           |2             |
+-------------------+-------------------+----------------+--------------+



In [9]:
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

# Query 3: Menulis data mentah ke file Parquet secara streaming
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

print("Streaming query 3 (Parquet sink) aktif:", query3.isActive)

Streaming query 3 (Parquet sink) aktif: True


In [10]:
# Mengirim batch 4 untuk menguji penulisan berkas Parquet
batch4 = pd.DataFrame({
    "pesanan_id": ["P013", "P014", "P015"],
    "nama_cabang": ["Kopi Nusantara Magelang", "Kopi Nusantara Semarang", "Kopi Nusantara Solo"],
    "nama_menu": ["Nasi Goreng Spesial", "Kentang Goreng", "Mocha"],
    "qty": [2, 2, 1],
    "harga_satuan": [30000, 15000, 28000],
    "waktu_pesan": [
        (waktu_dasar + timedelta(minutes=4)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar + timedelta(minutes=4)).strftime("%Y-%m-%d %H:%M:%S"),
        (waktu_dasar + timedelta(minutes=4)).strftime("%Y-%m-%d %H:%M:%S")
    ]
})

batch4.to_csv(f"{folder_stream}/batch_4.csv", index=False)
print("Batch 4 terkirim. Menunggu penulisan data ke berkas Parquet...")
time.sleep(8)

# Memeriksa isi folder tugas10_output
print("Berkas yang terbentuk di 'tugas10_output':")
for file_name in sorted(os.listdir(folder_output)):
    print(f" - {file_name}")

Batch 4 terkirim. Menunggu penulisan data ke berkas Parquet...


Berkas yang terbentuk di 'tugas10_output':
 - .part-00000-327c76e0-3e57-4bcd-ab4d-50f50979d8f8-c000.snappy.parquet.crc
 - .part-00001-2bfaa619-6cb8-475f-8046-342b1db3196f-c000.snappy.parquet.crc
 - .part-00002-a3b7891c-4a5b-4800-b270-c8b48dd598e6-c000.snappy.parquet.crc
 - .part-00003-2e18a64e-8e9a-4c40-ba3a-8f0ffe2611af-c000.snappy.parquet.crc
 - _spark_metadata
 - part-00000-327c76e0-3e57-4bcd-ab4d-50f50979d8f8-c000.snappy.parquet
 - part-00001-2bfaa619-6cb8-475f-8046-342b1db3196f-c000.snappy.parquet
 - part-00002-a3b7891c-4a5b-4800-b270-c8b48dd598e6-c000.snappy.parquet
 - part-00003-2e18a64e-8e9a-4c40-ba3a-8f0ffe2611af-c000.snappy.parquet


In [11]:
# Membaca kembali berkas Parquet sebagai DataFrame batch
df_verifikasi_parquet = spark.read.parquet(folder_output)

print("Skema DataFrame dari berkas Parquet:")
df_verifikasi_parquet.printSchema()

print(f"Total baris transaksi tersimpan: {df_verifikasi_parquet.count()}")
print("Data transaksi dari Parquet:")
df_verifikasi_parquet.show(truncate=False)

Skema DataFrame dari berkas Parquet:
root
 |-- pesanan_id: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- waktu_pesan: timestamp (nullable = true)



Total baris transaksi tersimpan: 15
Data transaksi dari Parquet:
+----------+-------------------------+-------------------+---+------------+-------------------+
|pesanan_id|nama_cabang              |nama_menu          |qty|harga_satuan|waktu_pesan        |
+----------+-------------------------+-------------------+---+------------+-------------------+
|P013      |Kopi Nusantara Magelang  |Nasi Goreng Spesial|2  |30000       |2026-11-11 20:10:05|
|P014      |Kopi Nusantara Semarang  |Kentang Goreng     |2  |15000       |2026-11-11 20:10:05|
|P015      |Kopi Nusantara Solo      |Mocha              |1  |28000       |2026-11-11 20:10:05|
|P005      |Kopi Nusantara Magelang  |Americano          |2  |18000       |2026-11-11 20:05:05|
|P006      |Kopi Nusantara Solo      |Mocha              |1  |28000       |2026-11-11 20:05:05|
|P007      |Kopi Nusantara Yogyakarta|Roti Bakar Cokelat |2  |18000       |2026-11-11 20:06:05|
|P008      |Kopi Nusantara Semarang  |Ayam Geprek        |1  |25000    

In [12]:
# Menghentikan seluruh streaming query
query_cabang.stop()
query_waktu.stop()
query3.stop()

print("Status aktif query_cabang:", query_cabang.isActive)
print("Status aktif query_waktu :", query_waktu.isActive)
print("Status aktif query3       :", query3.isActive)

# Menutup SparkSession
spark.stop()
print("SparkSession berhasil ditutup.")

Status aktif query_cabang: False
Status aktif query_waktu : False
Status aktif query3       : False
SparkSession berhasil ditutup.
